# 08 — Characterising the pre-Haiyan nighttime lights baseline

Estimate the normal level and variability of VIIRS DNB-BRDF in Samar–Leyte urban centres before 8 November 2013. Use the revised RQ1 mask and threshold as the starting point. Test whether the baseline depends on the radiance statistic, aggregation or viewing geometry.

| Section | Question |
|---|---|
| 1–2 | Which observations enter the analysis? |
| 3–4 | How intermittent is the record, and what drives the near-zero medians? |
| 5–6 | Do radiance and observability vary seasonally or with viewing geometry? |
| 7 | What baseline level and empirical range can the record support? |

Run in order from the repository root or its `notebooks/` folder. The pre-Haiyan record spans 19 January 2012 to 7 November 2013. The figures and tables are written to `output/baseline_cyclicity/`.

## 1. Inputs and analysis choices

| Input or setting | Role |
|---|---|
| Processed baseline and event-period VNP46A2 Zarr stores | DNB-BRDF and mandatory quality flag (MQF) |
| GHSL SMOD 2015 and municipal boundaries | Fixed raster support for G1 (GHSL class 30) |
| VNP46A1 stores, if present | Same-pixel, same-date sensor zenith |
| G1 with ≥25% daily SC | Primary regional configuration informed by revised RQ1 |
| MQF ∈ {0, 1} | HQ retrievals throughout the notebook |
| Last 60 pre-event days | Primary reference window, compared with 30, 90 and 180 days |

SC is the percentage of all fixed pixels in a settlement mask with finite eligible radiance on that date. An available scene with zero eligible pixels has 0% SC. An unavailable scene has undefined SC. No positive-radiance or baseline-completeness condition changes the fixed denominator.

RQ1 used a spatial mean and temporal P95 clipping of daily aggregate radiance. Here the primary mean uses a daily 0–P95 pixel cap, preserving the Notebook 08 convention. Section 7 includes the temporal-clipping variant for comparison. The reported RQ1 correlation motivates the choice of mask and window. This notebook estimates a pre-event radiance baseline.

“HQ retrievals” is the project label for the combined MQF 0 and 1 filter used here. The Collection 2 specification still distinguishes flags 0 and 1. [NASA VNP46A2 Collection 2 file specification](https://ladsweb.modaps.eosdis.nasa.gov/filespec/VIIRS/2/VNP46A2_v2.0.fs)

In [50]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
import geopandas as gpd
from rasterio.enums import Resampling
from rasterio.features import rasterize
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tqdm.auto import tqdm
from IPython.display import display

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if not (PROJECT_DIR / "datasets").exists(): PROJECT_DIR = Path('/Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery')
DATA_DIR = PROJECT_DIR / "datasets"
PROCESSED_DIR = DATA_DIR / "VNP46/processed"
BASELINE_STORE = PROCESSED_DIR / "baseline/Haiyan_VNP46A2_baseline.zarr"
EXISTING_STORE = PROCESSED_DIR / "Haiyan_VNP46A2.zarr"
if not BASELINE_STORE.exists() or not EXISTING_STORE.exists():
    raise FileNotFoundError(f"Expected both processed VNP46A2 stores under {PROCESSED_DIR}. Set PROJECT_DIR above.")
GHSL_PATH = next(DATA_DIR.glob("**/GHSL_SMOD_E2015.tif"), None)
if GHSL_PATH is None:
    raise FileNotFoundError(f"GHSL_SMOD_E2015.tif is missing under {DATA_DIR}.")
MUNICIPALITIES_PATH = DATA_DIR / "boundaries/MuniCities/MuniCities.shp"
OUTPUT_DIR = PROJECT_DIR / "output/baseline_cyclicity"
FIGURE_DIR, TABLE_DIR = OUTPUT_DIR / "figures", OUTPUT_DIR / "tables"
for path in (FIGURE_DIR,TABLE_DIR): path.mkdir(parents=True,exist_ok=True)
EVENT_DATE = pd.Timestamp("2013-11-08")
BASELINE_DAYS, AGGREGATION_DAYS = 60, 4
BASELINE_START = EVENT_DATE-pd.Timedelta(days=BASELINE_DAYS)
PRE_EVENT_END = EVENT_DATE-pd.Timedelta(days=1)
CHARACTERIZATION_START = pd.Timestamp("2012-01-19")

G1_CLASSES = (30,)
HQ_MQF = (0, 1)
SPATIAL_COMPLETENESS_PCT = 25.0
RQ_CLIP_PERCENTILE = 95.0
MIN_REFERENCE_DAYS = 10
LOW_NTL = .05
KNOWN_FILL_VALUES = (-9999., -999., -32768., 6553.5, 65535.)
DNB_BAND, MQF_BAND = "DNB_BRDF_Corrected_NTL", "Mandatory_Quality_Flag"
PLOT_FONT, PLOT_TEXT_COLOR, PLOT_GRID_COLOR = "Arial", "#243B5A", "#E8EDF3"
EVENT_COLOR, BASELINE_COLOR = "#0057FF", "#6C7882"
PRIMARY_COLOR, EXPANDED_COLOR, AGGREGATE_COLOR = "#0072B2", "#CA6400", "#008A61"
NTL_UNITS = "nW cm⁻² sr⁻¹"

In [51]:
def style_figure(fig,title,width=1280,height=720):
    fig.update_layout(template="plotly_white",paper_bgcolor="rgba(0,0,0,0)",plot_bgcolor="white",
        width=width,height=height,font=dict(family=PLOT_FONT,size=16,color=PLOT_TEXT_COLOR),
        title=dict(text=title,x=.03,xanchor="left",font=dict(size=23)),
        margin=dict(l=100,r=115,t=110,b=115),
        legend=dict(orientation="h",x=0,y=-.09,xanchor="left",yanchor="top"))
    fig.update_xaxes(zeroline=False,gridcolor=PLOT_GRID_COLOR)
    fig.update_yaxes(zeroline=False,gridcolor=PLOT_GRID_COLOR)
    return fig

def finish_figure(fig,filename):
    fig.write_html(FIGURE_DIR/f"{filename}.html",include_plotlyjs="directory")
    fig.show()

def add_haiyan_marker(fig,row=None,col=None):
    fig.add_vline(x=EVENT_DATE.to_pydatetime(),line_dash="dash",line_color=EVENT_COLOR,line_width=2,row=row,col=col)


def sc_heatmap(fig, coverage, row, col=1, bar_y=.14):
    fig.add_trace(go.Heatmap(
        x=coverage.index, y=coverage.columns,
        z=np.where(coverage.isna().T, 1., np.nan),
        colorscale=[[0, "#CFD5DC"], [1, "#CFD5DC"]], showscale=False,
        hoverongaps=False, hovertemplate="%{x|%d %b %Y} · %{y}<br>SC unavailable<extra></extra>",
    ), row=row, col=col)
    fig.add_trace(go.Heatmap(
        x=coverage.index, y=coverage.columns, z=coverage.T.to_numpy(), zmin=0, zmax=100,
        colorscale="Greens", zsmooth=False, hoverongaps=False,
        colorbar=dict(title="SC (%)", tickvals=[0, 50, 100], thickness=14, len=.22, y=bar_y),
        hovertemplate="%{x|%d %b %Y} · %{y}<br>SC %{z:.1f}%<extra></extra>",
    ), row=row, col=col)

def mark_reference(fig, rows):
    for row in rows:
        fig.add_vrect(x0=BASELINE_START, x1=EVENT_DATE, fillcolor=EVENT_COLOR,
                      opacity=.07, line_width=0, layer="below", row=row, col=1)
        add_haiyan_marker(fig, row=row, col=1)
    fig.add_annotation(x=EVENT_DATE, y=1.035, xref="x", yref="paper",
                       text="Haiyan · 8 Nov 2013", showarrow=False, xanchor="right",
                       font=dict(color=EVENT_COLOR, size=13))

def median(values, axis=None):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanmedian(values, axis=axis)

## 2. Build daily regional and settlement series

Read the stored calendar once. For G1 HQ retrievals, calculate the uncapped mean and median, mean after clipping to 0–P95, upper-tail radiance, exact-zero share and share near 0.010. Use the same fixed G1 pixels for every day. Calculate the daily statistics before applying the 25% SC gate so low-coverage dates can be inspected.

In [52]:
early = xr.open_zarr(BASELINE_STORE,chunks={},consolidated=False,mask_and_scale=False)
existing = xr.open_zarr(EXISTING_STORE,chunks={},consolidated=False,mask_and_scale=False)
early = early.sel(date=slice(CHARACTERIZATION_START, PRE_EVENT_END))
existing = existing.sel(date=slice(CHARACTERIZATION_START, PRE_EVENT_END))
if not np.all(early.processed.values == 1):
    raise RuntimeError("Complete the baseline A2 store for the pre-Haiyan dates.")
for axis in ("x","y"):
    np.testing.assert_allclose(early[axis].values,existing[axis].values,rtol=0,atol=1e-7)
early=early.assign_coords(x=existing.x,y=existing.y)
early["spatial_ref"]=existing["spatial_ref"]
complete=existing.processed.values==1
if len(pd.DatetimeIndex(existing.date.values[~complete]).difference(early.date.values)):
    raise RuntimeError("Existing event-period processing is unfinished.")
existing=existing.isel(date=np.flatnonzero(complete))
new_dates=pd.DatetimeIndex(early.date.values).difference(existing.date.values)
a2=xr.concat([early.sel(date=new_dates),existing],dim="date",data_vars="minimal",coords="minimal",compat="equals",join="exact").sortby("date")
a2=a2.rio.write_crs(existing.attrs["crs_wkt"])
template=a2[DNB_BAND].isel(date=0,drop=True)
ghsl=rxr.open_rasterio(GHSL_PATH,masked=True).squeeze("band",drop=True)
ghsl=ghsl.rio.reproject_match(template,resampling=Resampling.nearest).values
municipalities=gpd.read_file(MUNICIPALITIES_PATH)
region=municipalities[municipalities.PROVINCE.str.strip().str.title().isin(
    ["Samar","Eastern Samar","Northern Samar","Leyte","Southern Leyte"])].to_crs(a2.rio.crs).copy()
region["zone_id"]=np.arange(1,len(region)+1)
zone=rasterize([(g,int(i)) for g,i in zip(region.geometry,region.zone_id)],out_shape=ghsl.shape,
               transform=template.rio.transform(recalc=True),fill=0)


land = (zone > 0) & np.isin(ghsl, G1_CLASSES)
land_pixels = np.flatnonzero(land)
n_g1 = len(land_pixels)
if not n_g1:
    raise ValueError("No GHSL 30 pixels in the fixed Samar–Leyte footprint.")

daily_calendar = pd.date_range(CHARACTERIZATION_START, PRE_EVENT_END, name="date")
dates = pd.DatetimeIndex(a2.date.values)
if dates.empty:
    raise ValueError("No processed A2 scenes in the pre-Haiyan period.")
if not dates.is_unique:
    raise ValueError("The processed A2 stores contain duplicate daily dates.")
pixel_values = np.full((len(daily_calendar), n_g1), np.nan, dtype="float32")

records = []
for start in tqdm(range(0, len(dates), 8), desc="Read pre-Haiyan G1 HQ retrievals"):
    batch = a2[[DNB_BAND, MQF_BAND]].isel(date=slice(start, start + 8))
    batch = batch.transpose("date", "y", "x", missing_dims="ignore").compute()
    shape = (batch.sizes["date"], -1)
    radiance = batch[DNB_BAND].values.reshape(shape)[:, land_pixels].astype("float32")
    quality = batch[MQF_BAND].values.reshape(shape)[:, land_pixels]
    valid = (np.isfinite(radiance) & (radiance >= 0)
             & ~np.isin(radiance, KNOWN_FILL_VALUES) & np.isin(quality, HQ_MQF))
    radiance[~valid] = np.nan

    for date, day, keep, flags in zip(pd.DatetimeIndex(batch.date.values), radiance, valid, quality):
        pixel_values[daily_calendar.get_loc(date)] = day
        values = day[keep]
        sc = 100 * values.size / n_g1
        if values.size:
            cap = np.percentile(values, RQ_CLIP_PERCENTILE)
            stats = (values.mean(), np.clip(values, 0, cap).mean(), np.median(values), cap,
                     100 * np.mean(values == 0),
                     100 * np.isclose(values, .010, atol=1e-6, rtol=0).mean(),
                     100 * np.mean(flags[keep] == 1))
        else:
            stats = (np.nan,) * 7
        records.append((date, values.size, sc, *stats))

primary = pd.DataFrame(records, columns=[
    "date", "n", "SC", "mean", "mean_capped", "median", "P95",
    "zero_pct", "at_001_pct", "mqf1_pct",
]).set_index("date").reindex(daily_calendar)
qualified = primary.mean_capped.where(primary.SC.ge(SPATIAL_COMPLETENESS_PCT))
rolling = qualified.rolling(AGGREGATION_DAYS, center=True, min_periods=1)
window_n = rolling.count()
w4 = rolling.mean()
primary.assign(mask="G1", retrievals="HQ", qualified=qualified.notna()).to_csv(
    TABLE_DIR / "pre_haiyan_daily_diagnostics.csv"
)
print(f"Fixed G1 denominator: {n_g1:,} pixels. HQ retrievals: MQF 0 or 1.")

Read pre-Haiyan G1 HQ retrievals: 100%|██████████| 82/82 [00:01<00:00, 48.01it/s]


Fixed G1 denominator: 503 pixels. HQ retrievals: MQF 0 or 1.


## 3. Regional radiance and observability

The daily qualified mean supports the baseline. The centred four-day series uses calendar days and weights qualifying days equally. Its faint trace includes windows with one qualifying day. The stronger trace requires at least two. Both stop before Haiyan. The coverage strip uses green for 0–100% SC and grey for unavailable dates. Shading marks the final 60 days.

In [53]:
coverage = primary[["SC"]].rename(columns={"SC": "G1 · HQ retrievals"})
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, row_heights=[.52, .29, .19],
                    vertical_spacing=.09, subplot_titles=[
                        "G1 daily radiance and four-day mean",
                        "G1 daily spatial median", "Daily spatial completeness",
                    ])
fig.add_trace(go.Scatter(x=primary.index, y=primary.mean_capped, mode="markers",
    name="G1 below 25% SC", marker=dict(color="#AEB7C1", size=5),
    visible=True), row=1, col=1)
fig.data[-1].y = primary.mean_capped.where(primary.SC.lt(SPATIAL_COMPLETENESS_PCT))
fig.add_trace(go.Scatter(x=qualified.index, y=qualified, mode="markers", name="G1 daily qualified mean",
    marker=dict(color=PRIMARY_COLOR, size=6), customdata=primary[["SC", "n"]],
    hovertemplate="%{x|%d %b %Y}<br>%{y:.3f}<br>SC %{customdata[0]:.1f}%<br>%{customdata[1]:.0f} pixels<extra></extra>"),
    row=1, col=1)
for values, label, color, width in [(w4, "W=4 · ≥1 qualifying day", "#A4CCD7", 1),
                                   (w4.where(window_n.ge(2)), "W=4 · ≥2 qualifying days", "#00866D", 2.3)]:
    fig.add_trace(go.Scatter(x=values.index, y=values, name=label, mode="lines", connectgaps=False,
        line=dict(color=color, width=width), customdata=window_n,
        hovertemplate="%{x|%d %b %Y}<br>%{y:.3f}<br>%{customdata} qualifying days<extra></extra>"),
        row=1, col=1)
fig.add_trace(go.Scatter(x=primary.index, y=primary["median"], name="G1 daily median · HQ",
    mode="markers", marker=dict(color=PRIMARY_COLOR, size=5, opacity=.65),
    customdata=primary[["SC", "n"]],
    hovertemplate="%{x|%d %b %Y}<br>%{y:.3f}<br>SC %{customdata[0]:.1f}%<br>%{customdata[1]:.0f} pixels<extra></extra>"),
    row=2, col=1)
fig.add_hline(y=LOW_NTL, line_dash="dot", line_color="#8795A3", row=2, col=1)
sc_heatmap(fig, coverage, 3, bar_y=.10)
mark_reference(fig, (1, 2, 3))
for row in (1, 2):
    fig.update_yaxes(title_text=NTL_UNITS, rangemode="tozero", row=row, col=1)
fig.update_xaxes(range=[CHARACTERIZATION_START, EVENT_DATE+pd.Timedelta(days=2)], row=3, col=1)
style_figure(fig, "Pre-Haiyan radiance and observation support", 1420, 950)
finish_figure(fig, "01_pre_haiyan_ntl_and_sc")


status = pd.Series("Below 25% SC", index=daily_calendar, name="Status")
status.loc[primary.SC.isna()] = "No A2 scene"
status.loc[qualified.notna()] = "Qualified HQ date"
order = ["Qualified HQ date", "Below 25% SC", "No A2 scene"]
monthly_status = pd.crosstab(daily_calendar.to_period("M"), status).reindex(columns=order, fill_value=0)
fig = go.Figure()
for label, color in zip(order, [AGGREGATE_COLOR, "#C3AC89", "#CDD3DA"]):
    fig.add_trace(go.Bar(x=monthly_status.index.to_timestamp(), y=monthly_status[label],
                         name=label, marker_color=color))
fig.update_layout(barmode="stack")
fig.update_yaxes(title_text="Calendar days")
style_figure(fig, "Monthly availability at the 25% G1 coverage gate")
finish_figure(fig, "02_monthly_qualification")
monthly_status.to_csv(TABLE_DIR / "pre_haiyan_monthly_qualification.csv")
display(status.value_counts().reindex(order, fill_value=0).rename("Days").to_frame())

,Days
Status,
Qualified HQ date,289
Below 25% SC,365
No A2 scene,5


## 4. Diagnose low radiance

A median ≤0.05 nW cm⁻² sr⁻¹ is a diagnostic flag. Compare G1 HQ low medians with daily spatial completeness, upper-tail radiance and the share of pixels at 0.010. The audit records whether each flagged day satisfies the 25% gate.

In [54]:
fig = make_subplots(rows=2, cols=2, horizontal_spacing=.12, vertical_spacing=.15,
    subplot_titles=["Coverage and daily median", "Daily median and upper-tail radiance",
                    "Share at 0.010 and daily median", "Effect of the daily P95 cap on the mean"])
passes = primary.SC.ge(SPATIAL_COMPLETENESS_PCT)
for label, rows, color in [("Below 25% SC", primary.loc[~passes], "#AEB7C1"),
                           ("Qualified HQ date", primary.loc[passes], PRIMARY_COLOR)]:
    for row, col, x, y in [(1, 1, "SC", "median"), (1, 2, "median", "P95"),
                           (2, 1, "at_001_pct", "median"), (2, 2, "mean", "mean_capped")]:
        fig.add_trace(go.Scatter(
            x=rows[x], y=rows[y], name=label, legendgroup=label, showlegend=(row,col) == (1,1),
            mode="markers", marker=dict(size=6, color=color, opacity=.6),
            text=rows.index.strftime("%d %b %Y"),
            hovertemplate="%{text}<br>%{x:.3f} · %{y:.3f}<extra></extra>"), row=row, col=col)
fig.add_vline(x=SPATIAL_COMPLETENESS_PCT, line_dash="dot", line_color="#8795A3", row=1, col=1)
for row in (1, 2):
    fig.add_hline(y=LOW_NTL, line_dash="dot", line_color="#8795A3", row=row, col=1)
for row, col, label in [(1,1,"Spatial completeness (%)"), (1,2,f"Median ({NTL_UNITS})"),
                        (2,1,"Pixels at 0.010 (%)"), (2,2,f"Uncapped mean ({NTL_UNITS})")]:
    fig.update_xaxes(title_text=label, row=row,col=col)
fig.update_yaxes(title_text=NTL_UNITS)
style_figure(fig, "Low-radiance dates and clipping sensitivity · G1 HQ retrievals", 1420, 870)
finish_figure(fig, "03_low_radiance_diagnostics")

low_audit = primary.loc[primary["median"].le(LOW_NTL)].copy()
low_audit["qualified"] = low_audit.SC.ge(SPATIAL_COMPLETENESS_PCT)
low_audit.to_csv(TABLE_DIR / "pre_haiyan_near_zero_audit.csv")
print(f"{len(low_audit)} near-zero medians, including {low_audit.qualified.sum()} dates at ≥25% SC.")
display(low_audit.sort_values("SC", ascending=False).head(12).round(3))

6 near-zero medians, including 1 dates at ≥25% SC.


,n,SC,mean,mean_capped,median,P95,zero_pct,at_001_pct,mqf1_pct,qualified
date,,,,,,,,,,
2013-03-26,453.0,90.060,2.730,2.254,0.010,14.081,0.442,50.110,16.556,True
2012-08-30,121.0,24.056,1.061,0.978,0.010,4.894,0.000,57.025,14.876,False
2013-02-26,47.0,9.344,1.421,1.259,0.010,5.428,0.000,53.191,10.638,False
2012-08-02,33.0,6.561,1.834,1.790,0.010,6.253,0.000,57.576,18.182,False
2012-10-27,29.0,5.765,0.233,0.200,0.013,1.074,0.000,48.276,62.069,False
2012-09-04,9.0,1.789,0.208,0.202,0.011,0.610,22.222,22.222,0.000,False


## 5. Calendar structure and persistence

Use monthly distributions and a complete calendar to distinguish seasonal changes in radiance from seasonal changes in the number of qualifying observations. The lag diagnostic compares qualified G1 mean radiance and the binary qualification indicator at the same calendar-day separations. Each radiance lag uses observed pairs only and requires at least 20 pairs. These two partial years support an exploratory comparison of recurrence.

In [55]:
monthly = pd.DataFrame({"NTL": qualified, "SC": primary.SC}).groupby(daily_calendar.to_period("M"))
month = monthly.agg({"NTL": ["count", "median"], "SC": ["size", "median"]})
month.columns = ["Qualified days", "Median NTL", "Calendar days", "Median SC"]
month["Qualified (%)"] = 100 * month["Qualified days"] / month["Calendar days"]
month["Median NTL"] = month["Median NTL"].where(month["Qualified days"].ge(3))

fig = make_subplots(rows=2, cols=2, vertical_spacing=.13, horizontal_spacing=.13,
    subplot_titles=["Qualified daily radiance by month", "Qualification by calendar month",
                    "Calendar-lag correlation", "Paired days per radiance lag"])
for year, color in [(2012, PRIMARY_COLOR), (2013, EXPANDED_COLOR)]:
    days = qualified.loc[qualified.index.year == year].dropna()
    part = month.loc[month.index.year == year]
    fig.add_trace(go.Box(x=days.index.month, y=days, name=str(year), legendgroup=str(year),
                        marker_color=color, boxpoints=False), row=1,col=1)
    fig.add_trace(go.Scatter(x=part.index.month, y=part["Qualified (%)"], name=str(year),
        legendgroup=str(year), showlegend=False, mode="lines+markers", line=dict(color=color),
        customdata=part[["Qualified days", "Calendar days"]],
        hovertemplate="Month %{x}<br>%{y:.1f}% · %{customdata[0]}/%{customdata[1]} days<extra></extra>"),
        row=1,col=2)

lags = []
qualified_indicator = qualified.notna().astype(float)
for lag in range(1, 91):
    pairs = pd.concat([qualified, qualified.shift(lag)], axis=1).dropna()
    r = pairs.iloc[:, 0].corr(pairs.iloc[:, 1]) if len(pairs) >= 20 and pairs.std().gt(0).all() else np.nan
    indicator_r = qualified_indicator.autocorr(lag) if qualified_indicator.nunique() > 1 else np.nan
    lags.append((lag, len(pairs), r, indicator_r))
lags = pd.DataFrame(lags, columns=["lag", "n", "radiance_r", "qualification_r"])
for key, label, color in [("radiance_r", "Radiance", PRIMARY_COLOR),
                          ("qualification_r", "Qualification indicator", AGGREGATE_COLOR)]:
    fig.add_trace(go.Scatter(x=lags.lag, y=lags[key], name=label, line=dict(color=color)), row=2,col=1)
fig.add_trace(go.Bar(x=lags.lag, y=lags.n, marker_color="#A6B7C8", showlegend=False), row=2,col=2)
for col in (1,2):
    fig.update_xaxes(tickvals=list(range(1, 13)), ticktext=list("JFMAMJJASOND"), row=1,col=col)
    fig.update_xaxes(title_text="Lag (calendar days)", row=2,col=col)
fig.update_yaxes(title_text=NTL_UNITS, row=1,col=1)
fig.update_yaxes(title_text="Calendar days (%)", range=[0,105], row=1,col=2)
fig.update_yaxes(title_text="Pearson r", range=[-1,1], row=2,col=1)
fig.update_yaxes(title_text="Pairs", row=2,col=2)
fig.update_layout(boxmode="group")
style_figure(fig, "Seasonal support and short-term persistence", 1420, 900)
finish_figure(fig, "04_calendar_structure")
month.to_csv(TABLE_DIR / "pre_haiyan_monthly_support.csv")
lags.to_csv(TABLE_DIR / "pre_haiyan_calendar_lags.csv", index=False)

## 6. Match nighttime lights to measured viewing geometry

Compare A1 sensor zenith and A2 radiance at the same G1 pixel and date. Daily paired coverage uses the full G1 denominator. The angle-bin plot shows pixel-day distributions and the table compares low and high angles within the same pixel and month. A pixel-month needs at least two observations in each of the compared angle bands. This is an observational diagnostic for angle-related differences.

In [56]:
zenith_values = np.full(pixel_values.shape, np.nan, dtype="float32")
a1_dates = pd.DatetimeIndex([])
iy, ix = np.unravel_index(land_pixels, land.shape)
for path in (PROCESSED_DIR / "Haiyan_VNP46A1.zarr",
             PROCESSED_DIR / "baseline/Haiyan_VNP46A1_baseline.zarr"):
    if not path.exists():
        print(f"Missing A1 store: {path.name}")
        continue
    ds = xr.open_zarr(path, chunks={}, consolidated=False, mask_and_scale=False)
    for axis in ("x", "y"):
        np.testing.assert_allclose(ds[axis].values, a2[axis].values, rtol=0, atol=1e-6)
    band = ds["Sensor_Zenith"]
    for dim in set(band.dims) - {"date", "y", "x"}:
        if band.sizes[dim] != 1:
            raise ValueError(f"Unexpected Sensor_Zenith dimension: {dim}")
        band = band.isel({dim: 0}, drop=True)
    available = pd.DatetimeIndex(ds.date.values)
    if "processed" in ds:
        available = available[ds.processed.values == 1]
    selected_dates = daily_calendar.intersection(available).difference(a1_dates)
    a1_dates = a1_dates.union(selected_dates)
    fills = list(KNOWN_FILL_VALUES)
    for key in ("_FillValue", "missing_value"):
        if key in band.attrs:
            fills.extend(np.atleast_1d(band.attrs[key]).tolist())

    for start in tqdm(range(0, len(selected_dates), 16), desc=f"Read {path.name}"):
        subset = selected_dates[start:start + 16]
        values = band.sel(date=subset).isel(
            y=xr.DataArray(iy, dims="sample_pixel"), x=xr.DataArray(ix, dims="sample_pixel"),
        ).transpose("date", "sample_pixel").compute().values.astype("float32")
        values[np.isin(values, fills)] = np.nan
        values = values * float(band.attrs.get("scale_factor", 1)) + float(band.attrs.get("add_offset", 0))
        # Some legacy stores contain centidegrees without a scale attribute.
        finite = values[np.isfinite(values) & (values >= 0)]
        if "scale_factor" not in band.attrs and finite.size and np.percentile(finite, 95) > 90:
            values *= .01
        values[(values < 0) | (values > 90)] = np.nan
        zenith_values[daily_calendar.get_indexer(subset)] = values


pair = np.isfinite(zenith_values) & np.isfinite(pixel_values)
if pair.any():
    paired_ntl = np.where(pair, pixel_values, np.nan)
    paired_vza = np.where(pair, zenith_values, np.nan)
    paired_sc = pd.Series(100 * pair.mean(axis=1), index=daily_calendar)
    paired_sc = paired_sc.where(daily_calendar.isin(a1_dates) & primary.SC.notna())
    paired_day = pd.DataFrame({
        "median_ntl": median(paired_ntl, axis=1),
        "median_vza": median(paired_vza, axis=1),
        "SC": paired_sc,
    }, index=daily_calendar)
    paired_day.to_csv(TABLE_DIR / "pre_haiyan_matched_vza_daily.csv")
    fig = make_subplots(rows=3, cols=1, shared_xaxes=True, row_heights=[.42, .40, .18],
                        vertical_spacing=.10, subplot_titles=[
                            "Median NTL on paired pixels", "Median viewing zenith angle", "Paired G1 coverage",
                        ])
    enough = paired_sc.ge(SPATIAL_COMPLETENESS_PCT)
    for row, key, label in [(1, "median_ntl", NTL_UNITS), (2, "median_vza", "VZA (°)")]:
        fig.add_trace(go.Scatter(x=daily_calendar, y=paired_day[key].where(enough),
            mode="markers", marker=dict(color=PRIMARY_COLOR, size=6), showlegend=False), row=row,col=1)
        fig.update_yaxes(title_text=label, row=row,col=1)
    sc_heatmap(fig, paired_day[["SC"]].rename(columns={"SC": "VZA + NTL"}), 3, bar_y=.09)
    mark_reference(fig, (1,2,3))
    style_figure(fig, "Same-pixel viewing geometry and G1 radiance", 1420, 840)
    finish_figure(fig, "05_paired_viewing_geometry")

    bins = {"<30°": pair & (zenith_values < 30),
            "30–<50°": pair & (zenith_values >= 30) & (zenith_values < 50),
            "≥50°": pair & (zenith_values >= 50)}
    month_index = daily_calendar.to_period("M")
    log_ntl = np.log1p(paired_ntl)
    contrasts = []
    for month_id in month_index.unique():
        inside = month_index == month_id
        low = bins["<30°"][inside]
        low_median = median(np.where(low, log_ntl[inside], np.nan), axis=0)
        for label in ("30–<50°", "≥50°"):
            other = bins[label][inside]
            eligible = (low.sum(axis=0) >= 2) & (other.sum(axis=0) >= 2)
            other_median = median(np.where(other, log_ntl[inside], np.nan), axis=0)
            contrasts.extend((month_id.year, label, i, delta) for i, delta in
                             zip(np.flatnonzero(eligible), (other_median-low_median)[eligible]))
    fig = make_subplots(rows=1, cols=2, subplot_titles=[
        "Pooled pixel-days", "Within-pixel-month contrast with <30°",
    ])
    for label, eligible in bins.items():
        values = log_ntl[eligible]
        if values.size:
            quantiles = np.quantile(values, [.10,.25,.50,.75,.90])
            fig.add_trace(go.Box(name=label, q1=[quantiles[1]], median=[quantiles[2]],
                q3=[quantiles[3]], lowerfence=[quantiles[0]], upperfence=[quantiles[4]],
                marker_color=PRIMARY_COLOR, showlegend=False), row=1,col=1)
    if contrasts:
        vza_contrasts = pd.DataFrame(contrasts, columns=["year", "angle_band", "pixel", "delta_log1p"])
        per_pixel = vza_contrasts.groupby(["year", "angle_band", "pixel"], as_index=False).delta_log1p.median()
        vza_summary = per_pixel.groupby(["year","angle_band"]).delta_log1p.agg(["median","size"])
        display(vza_summary.rename(columns={"median": "Median Δ log(1 + NTL)", "size": "Pixels"}).round(3))
        vza_summary.to_csv(TABLE_DIR / "pre_haiyan_vza_contrasts.csv")
        for (year, label), part in per_pixel.groupby(["year", "angle_band"]):
            fig.add_trace(go.Box(y=part.delta_log1p, name=f"{year} · {label}",
                boxpoints=False, marker_color=PRIMARY_COLOR if year == 2012 else EXPANDED_COLOR,
                showlegend=False), row=1,col=2)
    else:
        fig.add_annotation(text="Insufficient paired observations", showarrow=False,
                           x=.5,y=.5,xref="x2 domain",yref="y2 domain")
    fig.add_hline(y=0, line_dash="dot", row=1,col=2)
    fig.update_yaxes(title_text="log(1 + NTL)", row=1,col=1)
    fig.update_yaxes(title_text="Δ log(1 + NTL)", row=1,col=2)
    style_figure(fig, "Radiance by viewing angle", 1360, 680)
    finish_figure(fig, "06_vza_angle_contrasts")
else:
    print("No overlapping G1 HQ radiance and valid A1 viewing angles.")

Read Haiyan_VNP46A1_baseline.zarr: 100%|██████████| 30/30 [00:00<00:00, 54.91it/s]


Median Δ log(1 + NTL)  Pixels
year angle_band                               
2012 30–<50°                     0.064     503
     ≥50°                        0.250     503
2013 30–<50°                     0.079     503
     ≥50°                        0.236     503

## 7. Characterise the baseline

The reference level is the median of the qualified daily G1 spatial means in the final 60 pre-event calendar days. P10–P90 describes day-to-day variability among the qualified observations. It is an empirical range. IQR, median absolute deviation and support counts describe the spread and sampling. A minimum of 10 qualifying days is used as a descriptive reporting criterion and is exposed in setup.

Compare four reference lengths, the four-day centred series with at least two qualifying days, pixel and temporal clipping, and the daily spatial median. For the temporal clipping sensitivity, estimate P95 across the available pre-Haiyan daily G1 means before applying the SC gate. Recompute each rolling series inside its reference window so it uses observations from that window only.

In [57]:
temporal_cap = primary["mean"].quantile(.95)
variants = {
    "G1 · daily mean": qualified,
    "G1 · temporal P95 mean": primary["mean"].clip(lower=0, upper=temporal_cap).where(primary.SC.ge(SPATIAL_COMPLETENESS_PCT)),
    "G1 · daily spatial median": primary["median"].where(primary.SC.ge(SPATIAL_COMPLETENESS_PCT)),
}
lookbacks = (30, BASELINE_DAYS, 90, 180)
reference_rows = []
reference_days = None
for days in sorted(set(lookbacks)):
    start = EVENT_DATE - pd.Timedelta(days=days)
    reference = {label: series.loc[start:PRE_EVENT_END] for label, series in variants.items()}
    daily_reference = reference["G1 · daily mean"]
    reference["G1 · W=4 (≥2 direct days)"] = daily_reference.rolling(
        AGGREGATION_DAYS, center=True, min_periods=2
    ).mean()
    if days == BASELINE_DAYS:
        reference_days = reference
    for label, values in reference.items():
        level = values.median()
        quantiles = values.quantile([.10,.25,.75,.90])
        n_direct = daily_reference.count() if "W=4" in label else values.count()
        reference_rows.append((days, label, values.count(), n_direct, level,
                               *quantiles, (values-level).abs().median()))
baseline = pd.DataFrame(reference_rows, columns=[
    "Lookback days", "Configuration", "Estimates", "Direct days", "Level",
    "P10", "P25", "P75", "P90", "MAD",
])
baseline["Direct fraction (%)"] = 100 * baseline["Direct days"] / baseline["Lookback days"]
baseline["Meets minimum support"] = baseline["Direct days"].ge(MIN_REFERENCE_DAYS)
baseline.to_csv(TABLE_DIR / "pre_haiyan_baseline_summary.csv", index=False)
display(baseline.set_index(["Lookback days","Configuration"]).round(3))

fig = make_subplots(rows=2, cols=1, vertical_spacing=.15, row_heights=[.56,.44],
                    subplot_titles=[f"Final {BASELINE_DAYS} days · daily observations and four-day mean",
                                    "Reference level across windows and configurations"])
final = baseline.loc[baseline["Lookback days"].eq(BASELINE_DAYS)].set_index("Configuration")
estimate = final.loc["G1 · daily mean"]
if estimate["Meets minimum support"]:
    fig.add_hrect(y0=estimate.P10, y1=estimate.P90, fillcolor=PRIMARY_COLOR,
                  opacity=.12, line_width=0, layer="below", row=1,col=1)
    fig.add_hline(y=estimate.Level, line_dash="dot", line_color=PRIMARY_COLOR, row=1,col=1)
for label, color, mode in [("G1 · daily mean", PRIMARY_COLOR, "markers"),
                           ("G1 · W=4 (≥2 direct days)", AGGREGATE_COLOR, "lines")]:
    values = reference_days[label]
    fig.add_trace(go.Scatter(x=values.index, y=values, name=label, mode=mode,
        marker=dict(color=color, size=7), line=dict(color=color), connectgaps=False), row=1,col=1)
for label, part in baseline.groupby("Configuration", sort=False):
    fig.add_trace(go.Scatter(x=part["Lookback days"], y=part.Level, name=label, mode="lines+markers",
        showlegend=label not in ("G1 · daily mean", "G1 · W=4 (≥2 direct days)"),
        line=dict(color={"G1 · daily mean": PRIMARY_COLOR,
                         "G1 · W=4 (≥2 direct days)": AGGREGATE_COLOR}.get(label)),
        marker=dict(symbol=np.where(part["Meets minimum support"], "circle", "circle-open")),
        customdata=part[["Direct days", "P10", "P90"]],
        hovertemplate="%{x} days<br>Level %{y:.3f}<br>%{customdata[0]} direct days"
                      "<br>P10–P90 %{customdata[1]:.3f}–%{customdata[2]:.3f}<extra></extra>"),
        row=2,col=1)
mark_reference(fig, (1,))
fig.update_xaxes(range=[BASELINE_START, EVENT_DATE+pd.Timedelta(days=1)], row=1,col=1)
fig.update_xaxes(title_text="Pre-event lookback (calendar days)", tickvals=sorted(set(lookbacks)), row=2,col=1)
fig.update_yaxes(title_text=NTL_UNITS)
style_figure(fig, "Pre-Haiyan reference level, variability and sensitivity", 1420, 900)
finish_figure(fig, "07_pre_haiyan_baseline")

n_ref = int(estimate["Direct days"])
if estimate["Meets minimum support"]:
    print(f"Primary pre-Haiyan G1 reference: {estimate.Level:.3f} {NTL_UNITS}. "
          f"Empirical P10–P90: {estimate.P10:.3f}–{estimate.P90:.3f} {NTL_UNITS}. "
          f"Based on {n_ref}/{BASELINE_DAYS} daily observations at ≥{SPATIAL_COMPLETENESS_PCT:g}% SC.")
    same_rule = baseline.loc[baseline.Configuration.eq("G1 · daily mean") & baseline["Meets minimum support"], "Level"]
    print(f"Across supported lookbacks, the G1 daily reference ranges from {same_rule.min():.3f} "
          f"to {same_rule.max():.3f} {NTL_UNITS}.")
else:
    print(f"Final {BASELINE_DAYS}-day reference has {n_ref} qualified days. "
          f"At least {MIN_REFERENCE_DAYS} are required by the reporting criterion. "
          "Review the longer-window estimates and their direct support.")

Estimates  Direct days  Level    P10  \
Lookback days Configuration                                                     
30            G1 · daily mean                    9            9  3.292  2.616   
              G1 · temporal P95 mean             9            9  3.757  2.898   
              G1 · daily spatial median          9            9  2.162  1.075   
              G1 · W=4 (≥2 direct days)         12            9  3.724  2.814   
60            G1 · daily mean                   14           14  3.244  2.611   
              G1 · temporal P95 mean            14           14  3.665  2.880   
              G1 · daily spatial median         14           14  1.875  1.279   
              G1 · W=4 (≥2 direct days)         17           14  3.755  3.035   
90            G1 · daily mean                   23           23  3.668  2.610   
              G1 · temporal P95 mean            23           23  4.211  2.873   
              G1 · daily spatial median         23           23  1.786  1.212   
              G1 · W=4 (≥2 direct days)         28           23  3.881  3.039   
180           G1 · daily mean                   53           53  3.644  2.364   
              G1 · temporal P95 mean            53           53  4.057  2.646   
              G1 · daily spatial median         53           53  1.774  0.527   
              G1 · W=4 (≥2 direct days)         62           53  3.813  2.780   

                                           P25    P75    P90    MAD  \
Lookback days Configuration                                           
30            G1 · daily mean            2.747  4.218  4.393  0.684   
              G1 · temporal P95 mean     3.017  4.429  4.984  0.740   
              G1 · daily spatial median  1.612  2.613  2.882  0.456   
              G1 · W=4 (≥2 direct days)  3.427  3.881  4.201  0.225   
60            G1 · daily mean            2.747  4.183  4.738  0.631   
              G1 · temporal P95 mean     2.962  4.418  5.574  0.730   
              G1 · daily spatial median  1.598  2.538  3.541  0.435   
              G1 · W=4 (≥2 direct days)  3.413  4.236  5.163  0.481   
90            G1 · daily mean            2.779  4.236  6.160  0.858   
              G1 · temporal P95 mean     3.080  4.590  6.599  1.067   
              G1 · daily spatial median  1.583  2.463  3.883  0.441   
              G1 · W=4 (≥2 direct days)  3.427  4.559  5.163  0.639   
180           G1 · daily mean            2.747  4.218  5.270  0.889   
              G1 · temporal P95 mean     2.996  4.705  6.013  1.040   
              G1 · daily spatial median  1.365  2.306  3.471  0.532   
              G1 · W=4 (≥2 direct days)  3.377  4.398  4.690  0.505   

                                         Direct fraction (%)  \
Lookback days Configuration                                    
30            G1 · daily mean                         30.000   
              G1 · temporal P95 mean                  30.000   
              G1 · daily spatial median               30.000   
              G1 · W=4 (≥2 direct days)               30.000   
60            G1 · daily mean                         23.333   
              G1 · temporal P95 mean                  23.333   
              G1 · daily spatial median               23.333   
              G1 · W=4 (≥2 direct days)               23.333   
90            G1 · daily mean                         25.556   
              G1 · temporal P95 mean                  25.556   
              G1 · daily spatial median               25.556   
              G1 · W=4 (≥2 direct days)               25.556   
180           G1 · daily mean                         29.444   
              G1 · temporal P95 mean                  29.444   
              G1 · daily spatial median               29.444   
              G1 · W=4 (≥2 direct days)               29.444   

                                         Meets minimum support  
Lookback days Configuration                                     
30            G1 · d

Primary pre-Haiyan G1 reference: 3.244 nW cm⁻² sr⁻¹. Empirical P10–P90: 2.611–4.738 nW cm⁻² sr⁻¹. Based on 14/60 daily observations at ≥25% SC.
Across supported lookbacks, the G1 daily reference ranges from 3.244 to 3.668 nW cm⁻² sr⁻¹.


### 7.1 Check when the reference is supported

Compare the final pre-event dates with the same calendar span in 2012. The aligned panels show whether the 60-day estimates use observations spread across the window and whether they are sensitive to year. The table reports the longest run of unqualified days and the last directly supported date in each window.

In [58]:
prior_start = BASELINE_START - pd.DateOffset(years=1)
prior_end = PRE_EVENT_END - pd.DateOffset(years=1)
windows = {
    "2012 · same calendar span": qualified.loc[prior_start:prior_end],
    "2013 · final reference window": qualified.loc[BASELINE_START:PRE_EVENT_END],
}
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[.72, .28],
                    subplot_titles=["Qualified G1 daily mean", "Daily G1 coverage"], vertical_spacing=.12)
seasonal_sc, window_checks = {}, []
for (label, values), color in zip(windows.items(), [EXPANDED_COLOR, PRIMARY_COLOR]):
    offsets = pd.date_range(BASELINE_START, periods=len(values))
    seasonal_sc[label] = pd.Series(primary.SC.reindex(values.index).to_numpy(), index=offsets)
    valid = values.notna()
    runs = valid.ne(valid.shift()).cumsum()
    longest_gap = int((~valid).groupby(runs).sum().max()) if len(valid) else 0
    window_checks.append((label, len(values), values.count(), longest_gap, values.last_valid_index(),
                          values.median(), values.quantile(.10), values.quantile(.90)))
    fig.add_trace(go.Scatter(x=offsets, y=values, name=label, mode="markers",
        marker=dict(color=color, size=7), text=values.index.strftime("%d %b %Y"),
        hovertemplate="%{text}<br>NTL %{y:.3f}<extra></extra>"), row=1,col=1)
sc_heatmap(fig, pd.DataFrame(seasonal_sc), 2, bar_y=.12)
fig.update_xaxes(tickformat="%d %b", title_text="Calendar date", row=2,col=1)
fig.update_yaxes(title_text=NTL_UNITS, row=1,col=1)
style_figure(fig, "Final reference window and same dates in 2012", 1390, 680)
finish_figure(fig, "08_reference_calendar_support")
window_checks = pd.DataFrame(window_checks, columns=[
    "Period", "Calendar days", "Direct days", "Longest unqualified run (days)",
    "Last qualifying date", "Median", "P10", "P90",
]).set_index("Period")
display(window_checks.round({key: 3 for key in ("Median", "P10", "P90")}))
window_checks.to_csv(TABLE_DIR / "pre_haiyan_reference_calendar_support.csv")

,Calendar days,Direct days,Longest unqualified run (days),Last qualifying date,Median,P10,P90
Period,,,,,,,
2012 · same calendar span,60,14,16,2012-11-06,3.936,2.640,6.181
2013 · final reference window,60,14,13,2013-11-02,3.244,2.611,4.738


## Conclusion

Carry forward the qualified G1 daily reference, its P10–P90 range and its observation count. The window comparison shows how much the reference level depends on the pre-event period. The clipping and reducer comparisons show how much it depends on radiance preprocessing and the statistic entering the estimate. If those choices give materially different levels, carry the variation into shock and recovery sensitivity.

Interpret this reference at urban-core scale. Use the four-day estimate to check temporal aggregation sensitivity and use the daily observations and their coverage to bound the timing of changes. A date with insufficient support remains observation-limited.

The `figures/` directory contains eight numbered interactive panels when A1 pairs are available. The `tables/` directory contains daily diagnostics, the near-zero date audit, monthly qualification and calendar summaries, VZA diagnostics when available, and `pre_haiyan_baseline_summary.csv`. Keep `plotly.min.js` beside the HTML files for offline viewing.